# Normalizing Activations in a Network

## 1. Core Idea

Normalizing activations in a neural network means applying normalization not only to the input data $X$, but also to intermediate values inside hidden layers.

Without Batch Normalization, a layer computes:

$$
Z^{[l]} = W^{[l]}A^{[l-1]} + b^{[l]}
$$

$$
A^{[l]} = g^{[l]}(Z^{[l]})
$$

With Batch Normalization, we normalize $Z^{[l]}$ before applying the activation function:

$$
Z^{[l]} \rightarrow Z_{\text{norm}}^{[l]} \rightarrow \tilde{Z}^{[l]} \rightarrow A^{[l]}
$$

Then:

$$
A^{[l]} = g^{[l]}(\tilde{Z}^{[l]})
$$

The main goal is to make the distribution of hidden-layer values more stable during training, so deeper layers do not need to constantly adapt to strongly changing inputs from previous layers.

## 2. BatchNorm Forward Propagation

For a mini-batch of size $m_b$:

$$
Z^{[l]} \in \mathbb{R}^{n^{[l]} \times m_b}
$$

BatchNorm computes the mini-batch mean:

$$
\mu^{[l]} = \frac{1}{m_b}\sum_{i=1}^{m_b} z^{[l](i)}
$$

and the mini-batch variance:

$$
(\sigma^2)^{[l]} =
\frac{1}{m_b}\sum_{i=1}^{m_b}
\left(z^{[l](i)}-\mu^{[l]}\right)^2
$$

Then it normalizes:

$$
Z_{\text{norm}}^{[l]} = \frac{Z^{[l]}-\mu^{[l]}}
{\sqrt{(\sigma^2)^{[l]}+\epsilon}}
$$

where $\epsilon$ is a small constant for numerical stability.

After normalization, BatchNorm applies a learnable scale and shift:

$$
\tilde{Z}^{[l]} = \gamma^{[l]}Z_{\text{norm}}^{[l]}+\beta^{[l]}
$$

Finally:

$$
A^{[l]} = g^{[l]}(\tilde{Z}^{[l]})
$$

## 3. Gamma, Beta, and Shape

The parameters $\gamma^{[l]}$ and $\beta^{[l]}$ are trainable parameters of BatchNorm.

They are usually initialized as:

$$
\gamma^{[l]} = 1
$$

$$
\beta^{[l]} = 0
$$

The role of $\gamma^{[l]}$ is to control the scale, and the role of $\beta^{[l]}$ is to control the shift:

$$
\tilde{Z}^{[l]} =
\gamma^{[l]}Z_{\text{norm}}^{[l]}+\beta^{[l]}
$$

They are necessary because the network should not be forced to always use zero-mean, unit-variance hidden values. BatchNorm first normalizes, then lets the network learn the most useful scale and shift.

Using Andrew Ng's notation:

$$
Z^{[l]} \in \mathbb{R}^{n^{[l]} \times m_b}
$$

$$
\mu^{[l]}, (\sigma^2)^{[l]}, \gamma^{[l]}, \beta^{[l]}
\in \mathbb{R}^{n^{[l]} \times 1}
$$

$$
Z_{\text{norm}}^{[l]}, \tilde{Z}^{[l]}
\in \mathbb{R}^{n^{[l]} \times m_b}
$$

In PyTorch, the convention is usually:

$$
Z \in \mathbb{R}^{m_b \times n^{[l]}}
$$

which is the transpose convention compared with Andrew Ng's notation.

## 4. Bias Term When Using BatchNorm

If BatchNorm is applied immediately after a linear layer, the bias term $b^{[l]}$ is usually unnecessary.

Normally:

$$
Z^{[l]} = W^{[l]}A^{[l-1]} + b^{[l]}
$$

But BatchNorm subtracts the mini-batch mean:

$$
Z^{[l]} - \mu^{[l]}
$$

Since $b^{[l]}$ is added equally to all examples in the mini-batch, it is removed when subtracting the mean.

Simplified:

$$
Z = WA + b
$$

$$
\mu = \text{mean}(WA+b)=\text{mean}(WA)+b
$$

Therefore:

$$
Z-\mu = WA-\text{mean}(WA)
$$

So the bias disappears during normalization.

BatchNorm already has its own shift parameter:

$$
\beta^{[l]}
$$

Therefore, when a linear layer is immediately followed by BatchNorm, we often use:

~~~python
nn.Linear(input_dim, hidden_dim, bias=False)
nn.BatchNorm1d(hidden_dim)
~~~

However, if a layer is not followed by BatchNorm, the bias term is still useful.

## 5. Backward Propagation Intuition

BatchNorm is differentiable, so it can be trained with backpropagation.

The forward path is:

$$
Z^{[l]}
\rightarrow
Z_{\text{norm}}^{[l]}
\rightarrow
\tilde{Z}^{[l]}
\rightarrow
A^{[l]}
$$

The backward path goes in reverse:

$$
dA^{[l]}
\rightarrow
d\tilde{Z}^{[l]}
\rightarrow
dZ_{\text{norm}}^{[l]}
\rightarrow
dZ^{[l]}
$$

Because:

$$
\tilde{Z}^{[l]}
= \gamma^{[l]}Z_{\text{norm}}^{[l]}+\beta^{[l]}
$$

BatchNorm learns gradients for $\gamma^{[l]}$ and $\beta^{[l]}$ as normal trainable parameters.

A useful intuition is:

- $\beta^{[l]}$ learns the best shift,
- $\gamma^{[l]}$ learns the best scale,
- $W^{[l]}$ is still learned through the usual linear transformation.

In practice, PyTorch and TensorFlow automatically compute the full BatchNorm backward pass.

## 6. Why BatchNorm Helps

BatchNorm mainly helps the **optimization process**. Its most important benefit is not simply making values look numerically nice, but making training more stable and easier for gradient-based optimization.

During training, the parameters of earlier layers keep changing. Therefore, the distribution of values received by deeper layers also changes. For example, layer $l$ receives $A^{[l-1]}$ from the previous layer. But $A^{[l-1]}$ depends on all earlier parameters:

$$
W^{[1]}, b^{[1]}, \dots, W^{[l-1]}, b^{[l-1]}
$$

As these parameters change, the input distribution of layer $l$ also changes. This makes optimization harder because each layer has to learn while its input distribution is constantly moving.

BatchNorm reduces this instability by normalizing intermediate values in each mini-batch:

$$
Z_{\text{norm}}^{[l]}
= \frac{Z^{[l]}-\mu^{[l]}}
{\sqrt{(\sigma^2)^{[l]}+\epsilon}}
$$

This gives the next activation function a more controlled input scale.

The main benefits are:

- **More stable training:** hidden-layer values are less likely to become extremely large or extremely small.
- **Faster convergence:** gradient descent often reaches a good solution in fewer iterations.
- **Larger learning rates may be possible:** because the intermediate values are normalized, training is less likely to diverge when using a moderately larger $\alpha$.
- **Less sensitivity to initialization:** even if the initial weights are not perfectly scaled, BatchNorm helps control the scale of hidden-layer values.
- **Reduced vanishing/exploding behavior:** by keeping intermediate values in a more stable range, BatchNorm helps gradients propagate more reliably.
- **Mild regularization effect:** mini-batch mean and variance introduce noise because each example is normalized using statistics from its current mini-batch.

However, BatchNorm should not be viewed as a complete replacement for L2 regularization, dropout, or data augmentation. Its primary role is to improve optimization, while its regularization effect is secondary.

## 7. Training Mode vs Inference Mode

BatchNorm behaves differently during training and inference.

During training, BatchNorm uses statistics from the current mini-batch:

$$
\mu_{\text{batch}}, \sigma^2_{\text{batch}}
$$

During inference, we may predict one example or a small batch, so current batch statistics may be unstable. Therefore, BatchNorm uses running estimates accumulated during training:

$$
\mu_{\text{running}}, \sigma^2_{\text{running}}
$$

In PyTorch:

~~~python
model.train()
~~~

means BatchNorm uses mini-batch statistics.

~~~python
model.eval()
~~~

means BatchNorm uses running statistics.

A common mistake is forgetting to call `model.eval()` during validation or testing.

## 8. Minimal Code Illustration

A common PyTorch pattern is:

~~~python
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(input_dim, hidden_dim, bias=False),
    nn.BatchNorm1d(hidden_dim),
    nn.ReLU(),

    nn.Linear(hidden_dim, output_dim)
)
~~~

BatchNorm is usually placed after the linear layer and before the activation function:

$$
\text{Linear} \rightarrow \text{BatchNorm} \rightarrow \text{Activation}
$$

The first linear layer often uses `bias=False` because BatchNorm already has a learnable shift parameter $\beta$.

## 9. Important Notes and Common Mistakes

A common mistake is thinking BatchNorm only normalizes. In fact, it normalizes and then learns scale and shift:

$$
\tilde{Z} = \gamma Z_{\text{norm}} + \beta
$$

Another mistake is confusing the BatchNorm parameter $\beta$ with the momentum or Adam hyperparameter $\beta$. They are different:

- BatchNorm $\beta$ is a trainable shift parameter,
- Momentum/Adam $\beta$ is a hyperparameter.

Another important point is that the bias term before BatchNorm is usually redundant, because BatchNorm subtracts the mean and already has $\beta$.

During validation or testing, forgetting `model.eval()` can produce unstable results because BatchNorm may use mini-batch statistics instead of running statistics.

Finally, BatchNorm can be less effective with very small batch sizes because the mini-batch mean and variance become noisy.

## 10. Essential Conclusion

The main idea of **Normalizing Activations in a Network** is:

> Normalize hidden-layer values so that deeper layers receive more stable inputs during training.

BatchNorm changes the hidden-layer computation from:

$$
Z^{[l]} \rightarrow A^{[l]}
$$

to:

$$
Z^{[l]}
\rightarrow
Z_{\text{norm}}^{[l]}
\rightarrow
\tilde{Z}^{[l]}
\rightarrow
A^{[l]}
$$

The two core equations are:

$$
Z_{\text{norm}}^{[l]}
=
\frac{Z^{[l]}-\mu^{[l]}}
{\sqrt{(\sigma^2)^{[l]}+\epsilon}}
$$

$$
\tilde{Z}^{[l]}
=
\gamma^{[l]}Z_{\text{norm}}^{[l]}
+
\beta^{[l]}
$$

The essence is:

$$
\boxed{
\text{BatchNorm normalizes hidden values, then learns how to scale and shift them.}
}
$$

# Fitting Batch Norm into a Neural Network

## 1. Core Idea

Batch Normalization is inserted into a neural network between the linear step and the activation function.

Without BatchNorm, a hidden layer usually computes:

$$
Z^{[l]} = W^{[l]}A^{[l-1]} + b^{[l]}
$$

$$
A^{[l]} = g^{[l]}(Z^{[l]})
$$

With BatchNorm, we normalize the linear output before applying the activation function:

$$
Z^{[l]} \rightarrow Z_{\text{norm}}^{[l]} \rightarrow \tilde{Z}^{[l]} \rightarrow A^{[l]}
$$

The common pattern is:

$$
\boxed{
\text{Linear} \rightarrow \text{BatchNorm} \rightarrow \text{Activation}
}
$$

If Dropout is used, a common hidden block is:

$$
\boxed{
\text{Linear} \rightarrow \text{BatchNorm} \rightarrow \text{Activation} \rightarrow \text{Dropout}
}
$$

The purpose of adding BatchNorm is to make hidden-layer values more stable during training, which usually makes optimization easier and more reliable.

## 2. Forward Propagation with BatchNorm

For a hidden layer, the computation becomes:

$$
Z^{[l]} = W^{[l]}A^{[l-1]}
$$

Then BatchNorm computes the mini-batch mean and variance:

$$
\mu^{[l]} =
\frac{1}{m_b}
\sum_{i=1}^{m_b}
z^{[l](i)}
$$

$$
(\sigma^2)^{[l]} =
\frac{1}{m_b}
\sum_{i=1}^{m_b}
\left(z^{[l](i)}-\mu^{[l]}\right)^2
$$

Then it normalizes:

$$
Z_{\text{norm}}^{[l]}
= \frac{Z^{[l]}-\mu^{[l]}}
{\sqrt{(\sigma^2)^{[l]}+\epsilon}}
$$

where $\epsilon$ is a small constant used for numerical stability.

After normalization, BatchNorm applies a learnable scale and shift:

$$
\tilde{Z}^{[l]}
= \gamma^{[l]}Z_{\text{norm}}^{[l]}+\beta^{[l]}
$$

Finally, the activation function is applied:

$$
A^{[l]} = g^{[l]}(\tilde{Z}^{[l]})
$$

So BatchNorm does not replace the activation function. It is inserted before the activation.

## 3. Parameters Learned with BatchNorm

Without BatchNorm, a hidden layer usually learns:

$$
W^{[l]}, b^{[l]}
$$

With BatchNorm immediately after the linear step, the bias term $b^{[l]}$ is usually unnecessary.

The reason is that BatchNorm subtracts the mini-batch mean. Since the bias is added equally to all examples in the mini-batch, it is removed during mean subtraction.

Simplified:

$$
Z = WA + b
$$

$$
\mu = \text{mean}(WA+b)=\text{mean}(WA)+b
$$

Therefore:

$$
Z-\mu = WA-\text{mean}(WA)
$$

So the usual bias term disappears during normalization.

BatchNorm introduces two new trainable parameters:

$$
\gamma^{[l]}, \beta^{[l]}
$$

where:

- $\gamma^{[l]}$ controls the scale,
- $\beta^{[l]}$ controls the shift,
- $\gamma^{[l]}$ is usually initialized to $1$,
- $\beta^{[l]}$ is usually initialized to $0$.

Therefore, a BatchNorm hidden layer usually learns:

$$
W^{[l]}, \gamma^{[l]}, \beta^{[l]}
$$

instead of:

$$
W^{[l]}, b^{[l]}
$$

Important note:

> $\beta^{[l]}$ in BatchNorm is a trainable shift parameter. It is not the same as the $\beta$ hyperparameter in Momentum or Adam.

## 4. Backward Propagation Intuition

Since BatchNorm is inserted between the linear step and the activation function, the forward path is:

$$
Z^{[l]}
\rightarrow
Z_{\text{norm}}^{[l]}
\rightarrow
\tilde{Z}^{[l]}
\rightarrow
A^{[l]}
$$

The backward path goes in the reverse direction:

$$
dA^{[l]}
\rightarrow
d\tilde{Z}^{[l]}
\rightarrow
dZ_{\text{norm}}^{[l]}
\rightarrow
dZ^{[l]}
$$

BatchNorm also produces gradients for its own trainable parameters:

$$
d\gamma^{[l]}, d\beta^{[l]}
$$

After obtaining $dZ^{[l]}$, we compute the usual gradient for the linear layer:

$$
dW^{[l]}
$$

The full backward computation through BatchNorm is algebraically more complex because $Z_{\text{norm}}^{[l]}$ depends on $Z^{[l]}$, the mini-batch mean, and the mini-batch variance.

In practice, frameworks such as PyTorch and TensorFlow compute this automatically.

The key idea is:

> BatchNorm is fully differentiable, so $W^{[l]}$, $\gamma^{[l]}$, and $\beta^{[l]}$ can all be learned by backpropagation.

## 5. Training Mode and Inference Mode

BatchNorm behaves differently during training and inference.

During training, BatchNorm uses statistics from the current mini-batch:

$$
\mu_{\text{batch}}, \sigma^2_{\text{batch}}
$$

During inference, we may predict one example or a very small batch. In that case, the current batch statistics may be unstable.

Therefore, during training, BatchNorm keeps running estimates:

$$
\mu_{\text{running}}, \sigma^2_{\text{running}}
$$

During inference, it uses these running statistics instead of the current batch statistics.

In PyTorch:

~~~python
model.train()
~~~

uses mini-batch statistics.

~~~python
model.eval()
~~~

uses running statistics.

A common mistake is forgetting to call `model.eval()` during validation or testing.

## 6. Minimal PyTorch Example

A basic hidden block with BatchNorm is:

~~~python
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(input_dim, hidden_dim, bias=False),
    nn.BatchNorm1d(hidden_dim),
    nn.ReLU(),

    nn.Linear(hidden_dim, output_dim)
)
~~~

Here:

- `Linear` computes the linear transformation,
- `BatchNorm1d` normalizes the hidden values,
- `ReLU` applies the activation function,
- `bias=False` is used because BatchNorm already has the shift parameter $\beta$.

If Dropout is added, it is commonly placed after the activation:

~~~python
model = nn.Sequential(
    nn.Linear(input_dim, hidden_dim, bias=False),
    nn.BatchNorm1d(hidden_dim),
    nn.ReLU(),
    nn.Dropout(p=0.2),

    nn.Linear(hidden_dim, output_dim)
)
~~~

## 7. Important Notes

BatchNorm is usually applied to hidden layers, not necessarily to the output layer. The output layer often needs to produce logits or predictions directly for the loss function.

BatchNorm changes the forward path of the network, while L2 regularization changes the objective function by adding a penalty to the loss. Therefore, BatchNorm and L2 regularization play different roles.

If the mini-batch size is too small, BatchNorm statistics can become noisy because the mini-batch mean and variance are not reliable.

BatchNorm can make training more stable, allow larger learning rates, reduce sensitivity to initialization, and sometimes provide a mild regularization effect. However, it should not be treated as a complete replacement for L2 regularization, Dropout, or data augmentation.

## 8. Essential Conclusion

The main idea of **Fitting Batch Norm into a Neural Network** is that BatchNorm is inserted into hidden layers between the linear transformation and the activation function.

Without BatchNorm:

$$
Z^{[l]} \rightarrow A^{[l]}
$$

With BatchNorm:

$$
Z^{[l]}
\rightarrow
Z_{\text{norm}}^{[l]}
\rightarrow
\tilde{Z}^{[l]}
\rightarrow
A^{[l]}
$$

A BatchNorm hidden layer usually learns:

$$
W^{[l]}, \gamma^{[l]}, \beta^{[l]}
$$

instead of:

$$
W^{[l]}, b^{[l]}
$$

because the usual bias term is often redundant when BatchNorm immediately follows the linear step.

The essence is:

$$
\boxed{
\text{BatchNorm is fitted into a network as }
\text{Linear} \rightarrow \text{BatchNorm} \rightarrow \text{Activation}.
}
$$

It helps make hidden-layer values more stable, which makes optimization easier and training more reliable.

# Why Does Batch Norm Work?

## 1. Core Idea

Batch Normalization works because it makes the training of deep neural networks more stable and easier to optimize.

In a deep network, the input distribution of each hidden layer changes during training because the parameters of earlier layers keep changing. For layer $l$, the input depends on previous activations:

$$
Z^{[l]} = W^{[l]}A^{[l-1]} + b^{[l]}
$$

If the distribution of $A^{[l-1]}$ changes a lot, then layer $l$ has to keep adapting to a moving input distribution.

BatchNorm reduces this instability by normalizing the intermediate values:

$$
Z_{\text{norm}}^{[l]}
= \frac{Z^{[l]}-\mu^{[l]}}
{\sqrt{(\sigma^2)^{[l]}+\epsilon}}
$$

Then it restores flexibility using learnable scale and shift parameters:

$$
\tilde{Z}^{[l]}
= \gamma^{[l]}Z_{\text{norm}}^{[l]}+\beta^{[l]}
$$

So BatchNorm does two things at the same time:

- it stabilizes hidden-layer values,
- it still lets the network learn the best scale and shift.

## 2. More Stable Optimization

The main benefit of BatchNorm is improved optimization.

Without BatchNorm, hidden values can become too large, too small, or shift too much during training. This can make gradient descent unstable.

For example, with sigmoid or tanh, very large positive or negative values can push activations into saturation regions, where gradients become very small.

With ReLU, if many pre-activation values become negative, many neurons may become inactive.

BatchNorm keeps intermediate values in a more controlled range before activation. This helps gradients propagate more reliably through the network.

As a result, BatchNorm often leads to:

- more stable training,
- faster convergence,
- smoother learning curves,
- fewer exploding or vanishing activation problems.

The key point is:

> BatchNorm does not simply make numbers look nicer. It makes the optimization problem easier.

## 3. Larger Learning Rates and Less Sensitivity to Initialization

BatchNorm often allows the use of larger learning rates.

A large learning rate can make training faster, but it can also cause the loss to oscillate or diverge. By keeping hidden-layer values more stable, BatchNorm often makes the network more tolerant of moderately larger learning rates.

This does not mean any learning rate will work. It means BatchNorm usually expands the range of learning rates that train successfully.

BatchNorm also reduces sensitivity to initialization.

Without BatchNorm, if initial weights are too large, hidden values can become too large. If initial weights are too small, signals and gradients can become weak. BatchNorm helps control the scale of intermediate values, so training becomes less dependent on perfectly chosen initial weights.

Initialization is still important, but BatchNorm makes the network more robust.

## 4. Mild Regularization Effect

BatchNorm can also have a mild regularization effect.

During training, BatchNorm computes mean and variance from the current mini-batch:

$$
\mu_{\text{batch}}, \sigma^2_{\text{batch}}
$$

Different mini-batches produce slightly different statistics. Therefore, the normalized value of one example depends slightly on the other examples in the same mini-batch.

This introduces noise into training, which can reduce overfitting slightly.

However, BatchNorm should not be treated as a full replacement for:

- L2 regularization,
- dropout,
- data augmentation,
- early stopping.

Its primary role is optimization. Its regularization effect is secondary.

## 5. Internal Covariate Shift and Modern View

In Andrew Ng's course, BatchNorm is explained using the idea of **internal covariate shift**.

The intuition is that as earlier layers update, the distribution of activations in later layers changes. BatchNorm reduces this shifting distribution by normalizing hidden-layer values.

This is a useful explanation for understanding the motivation.

A more optimization-focused view is that BatchNorm works because it makes the loss landscape easier to optimize and gradients more stable.

So we can understand BatchNorm at two levels:

- Course intuition: it reduces changes in hidden-layer input distributions.
- Optimization intuition: it stabilizes values and gradients, making training easier.

## 6. Essential Conclusion

BatchNorm works because it stabilizes hidden-layer values during training.

It transforms:

$$
Z^{[l]}
\rightarrow
Z_{\text{norm}}^{[l]}
\rightarrow
\tilde{Z}^{[l]}
$$

with:

$$
Z_{\text{norm}}^{[l]}
= \frac{Z^{[l]}-\mu^{[l]}}
{\sqrt{(\sigma^2)^{[l]}+\epsilon}}
$$

and:

$$
\tilde{Z}^{[l]}
= \gamma^{[l]}Z_{\text{norm}}^{[l]}+\beta^{[l]}
$$

The main benefits are:

- hidden-layer distributions become more stable,
- optimization becomes easier,
- gradients become more reliable,
- larger learning rates may be possible,
- training becomes less sensitive to initialization,
- there is a mild regularization effect.

The essence is:

$$
\boxed{
\text{BatchNorm works because it stabilizes hidden-layer values and makes optimization easier.}
}
$$

# Batch Norm at Test Time

## 1. Core Idea

During training, Batch Normalization uses the mean and variance of the current mini-batch:

$$
\mu_{\text{batch}}, \sigma^2_{\text{batch}}
$$

For a BatchNorm layer, the training-time normalization is:

$$
Z_{\text{norm}}
=
\frac{Z-\mu_{\text{batch}}}
{\sqrt{\sigma^2_{\text{batch}}+\epsilon}}
$$

Then BatchNorm applies the learned scale and shift:

$$
\tilde{Z}
=
\gamma Z_{\text{norm}}+\beta
$$

The issue is that at test time, we may predict one example or a very small batch. In that case, the current batch mean and variance may be unreliable.

Therefore, at test time, BatchNorm does not use the test batch statistics. Instead, it uses running statistics accumulated during training.

## 2. Running Mean and Running Variance

During training, each mini-batch produces its own statistics:

$$
\mu_{\text{batch}}^{(t)}, \sigma_{\text{batch}}^{2(t)}
$$

Because different mini-batches contain different examples, these values change from batch to batch.

To make inference stable, BatchNorm maintains running estimates:

$$
\mu_{\text{running}}, \sigma^2_{\text{running}}
$$

A common update form is:

$$
\mu_{\text{running}}
:=
\rho \mu_{\text{running}}
+
(1-\rho)\mu_{\text{batch}}^{(t)}
$$

$$
\sigma^2_{\text{running}}
:=
\rho \sigma^2_{\text{running}}
+
(1-\rho)\sigma_{\text{batch}}^{2(t)}
$$

These running statistics approximate the mean and variance of the training distribution.

At test time, BatchNorm uses:

$$
Z_{\text{norm}}
=
\frac{Z-\mu_{\text{running}}}
{\sqrt{\sigma^2_{\text{running}}+\epsilon}}
$$

Then:

$$
\tilde{Z}
=
\gamma Z_{\text{norm}}+\beta
$$

The key difference is:

```text
Training: use mini-batch mean and variance.
Test time: use running mean and running variance.
```

## 3. What Happens to Gamma and Beta at Test Time?

The parameters $\gamma$ and $\beta$ are different from running mean and running variance.

Running mean and running variance are statistics accumulated from mini-batches. They are not learned by gradient descent.

But $\gamma$ and $\beta$ are trainable parameters of each BatchNorm layer.

They are learned during training by backpropagation:

$$
\gamma := \gamma - \alpha d\gamma
$$

$$
\beta := \beta - \alpha d\beta
$$

At test time, BatchNorm still uses the learned $\gamma$ and $\beta$.

For each BatchNorm layer, test-time computation is:

$$
Z_{\text{norm}}^{[l]}
=
\frac{
Z^{[l]}-\mu_{\text{running}}^{[l]}
}
{
\sqrt{(\sigma^2_{\text{running}})^{[l]}+\epsilon}
}
$$

$$
\tilde{Z}^{[l]}
=
\gamma^{[l]}Z_{\text{norm}}^{[l]}+\beta^{[l]}
$$

Each BatchNorm layer has its own:

$$
\gamma^{[l]}, \beta^{[l]},
\mu_{\text{running}}^{[l]},
(\sigma^2_{\text{running}})^{[l]}
$$

So at test time, each layer uses its own learned scale, learned shift, running mean, and running variance.

## 4. Why Not Use Test Batch Statistics?

There are three main reasons.

First, the test batch may be very small. If batch size is $1$, the variance estimate is not meaningful.

Second, prediction for one example should not depend on which other examples happen to be in the same test batch.

Third, using running statistics makes inference consistent and stable.

So BatchNorm at test time uses statistics learned from training instead of statistics computed from the current test batch.

## 5. PyTorch Behavior

In PyTorch, BatchNorm behaves differently depending on the model mode.

During training:

~~~python
model.train()
~~~

BatchNorm:

```text
uses current mini-batch statistics
updates running mean and running variance
```

During validation, testing, or inference:

~~~python
model.eval()
~~~

BatchNorm:

```text
uses running mean and running variance
does not update running statistics
still uses learned gamma and beta
```

A common mistake is forgetting to call:

~~~python
model.eval()
~~~

before validation or testing.

This can make validation or test results unstable because BatchNorm may continue using batch statistics instead of running statistics.

## 6. Important Distinction

There are two different mechanisms in BatchNorm.

First, running statistics:

$$
\mu_{\text{running}}, \sigma^2_{\text{running}}
$$

These are updated by moving averages during training and used at test time.

Second, trainable parameters:

$$
\gamma, \beta
$$

These are learned by gradient descent and used in both training and test time.

So:

```text
running_mean, running_var:
- statistics
- updated by moving average
- used for stable inference

gamma, beta:
- trainable parameters
- updated by backpropagation
- used in both training and inference
```

## 7. Essential Conclusion

BatchNorm behaves differently during training and test time.

During training:

$$
Z_{\text{norm}}
=
\frac{Z-\mu_{\text{batch}}}
{\sqrt{\sigma^2_{\text{batch}}+\epsilon}}
$$

During test time:

$$
Z_{\text{norm}}
=
\frac{Z-\mu_{\text{running}}}
{\sqrt{\sigma^2_{\text{running}}+\epsilon}}
$$

In both cases, BatchNorm still applies the learned scale and shift:

$$
\tilde{Z}
=
\gamma Z_{\text{norm}}+\beta
$$

The essence is:

$$
\boxed{
\text{Training uses mini-batch statistics; test time uses running statistics.}
}
$$

Each BatchNorm layer uses its own learned $\gamma$, learned $\beta$, running mean, and running variance.

# Softmax Regression

## 1. Core Idea

Softmax Regression is the extension of logistic regression from binary classification to multi-class classification.

Binary classification predicts one probability:

$$
\hat{y} = P(y=1 \mid x)
$$

Softmax Regression predicts a probability distribution over $C$ classes:

$$
\hat{y}
=
\begin{bmatrix}
\hat{y}_1 \\
\hat{y}_2 \\
\vdots \\
\hat{y}_C
\end{bmatrix}
$$

where:

$$
0 \leq \hat{y}_j \leq 1
$$

and:

$$
\sum_{j=1}^{C}\hat{y}_j = 1
$$

The class with the largest probability is usually selected as the prediction:

$$
\hat{c} = \arg\max_j \hat{y}_j
$$

Softmax is used when each example belongs to exactly one class.

## 2. Logits and Softmax Function

Before Softmax, the output layer produces raw scores called logits:

$$
z =
\begin{bmatrix}
z_1 \\
z_2 \\
\vdots \\
z_C
\end{bmatrix}
$$

These logits are not probabilities. They can be negative, greater than $1$, and do not necessarily sum to $1$.

Softmax converts logits into probabilities:

$$
\hat{y}_j =
\frac{e^{z_j}}
{\sum_{k=1}^{C} e^{z_k}}
$$

for:

$$
j = 1,2,\dots,C
$$

Each logit $z_j$ represents the score for class $j$. The exponential makes the score positive, and the denominator normalizes all scores so that the probabilities sum to $1$.

The model output is:

$$
A^{[L]} = \text{softmax}(Z^{[L]})
$$

## 3. Cross-Entropy Loss

For one example, the true label $y$ is usually represented as a one-hot vector.

Example with $3$ classes:

$$
y =
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

The cross-entropy loss is:

$$
L(\hat{y}, y)
=
-\sum_{j=1}^{C} y_j \log(\hat{y}_j)
$$

Because $y$ is one-hot, only the probability of the correct class matters. If the correct class is $c$, then:

$$
L(\hat{y}, y)
=
-\log(\hat{y}_c)
$$

If the model gives high probability to the correct class, the loss is small. If it gives low probability to the correct class, the loss is large.

For $m$ examples:

$$
J =
-\frac{1}{m}
\sum_{i=1}^{m}
\sum_{j=1}^{C}
y_j^{(i)}\log(\hat{y}_j^{(i)})
$$

## 4. Backward Propagation

A very important result is that Softmax combined with Cross-Entropy gives a simple output-layer gradient:

$$
dZ^{[L]} = A^{[L]} - Y
$$

This means the gradient is simply the difference between predicted probabilities and true labels.

For example:

$$
A =
\begin{bmatrix}
0.1 \\
0.8 \\
0.1
\end{bmatrix}
$$

and:

$$
Y =
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

Then:

$$
dZ =
A - Y
=
\begin{bmatrix}
0.1 \\
-0.2 \\
0.1
\end{bmatrix}
$$

The model will decrease the logits of incorrect classes and increase the logit of the correct class.

## 5. Numerical Stability

When implementing Softmax directly, we should avoid numerical overflow.

Instead of computing:

$$
\frac{e^{z_j}}
{\sum_k e^{z_k}}
$$

directly, we subtract the maximum logit first:

$$
\text{softmax}(z)_j =
\frac{e^{z_j-\max(z)}}
{\sum_k e^{z_k-\max(z)}}
$$

This does not change the Softmax result, but it makes computation more stable.

Minimal NumPy implementation:

~~~python
import numpy as np

def softmax(Z):
    Z_shifted = Z - np.max(Z, axis=1, keepdims=True)
    exp_Z = np.exp(Z_shifted)
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)
~~~

Here, `Z` has shape `(m, C)`, where `m` is the number of examples and `C` is the number of classes.

## 6. PyTorch and TensorFlow Usage

In PyTorch, `nn.CrossEntropyLoss()` expects raw logits, not Softmax probabilities.

Correct usage:

~~~python
import torch.nn as nn

logits = model(X)
loss = nn.CrossEntropyLoss()(logits, y)
~~~

Do not apply Softmax before `CrossEntropyLoss`, because PyTorch already combines `LogSoftmax` and negative log-likelihood loss internally.

In TensorFlow/Keras, one common and stable approach is to output logits and set `from_logits=True`:

~~~python
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Dense(num_classes)
])

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
~~~

If the final layer already uses `activation="softmax"`, then `from_logits=False` should be used.

## 7. Softmax vs Sigmoid

Softmax is used for multi-class single-label classification.

Example:

```text
One image is either cat, dog, or horse.
```

The classes compete with each other, and the probabilities sum to $1$.

Sigmoid is used for multi-label classification.

Example:

```text
One image may contain person, car, and dog at the same time.
```

In that case, each class is treated as an independent binary classification problem.

The rule is:

```text
Multi-class single-label → Softmax
Multi-label → Sigmoid per class
```

## 8. Important Notes

Softmax does not choose the class by itself. It produces probabilities. The predicted class is usually selected using:

$$
\arg\max_j \hat{y}_j
$$

The output of Softmax should be interpreted as a probability distribution over classes.

A common mistake in PyTorch is applying Softmax before `nn.CrossEntropyLoss()`. This is unnecessary and can make training less numerically stable.

Another common mistake is using Softmax for multi-label classification. Softmax assumes that exactly one class is correct, so it is not suitable when multiple classes can be true at the same time.

## 9. Essential Conclusion

Softmax Regression is used for multi-class classification.

It converts logits into a probability distribution:

$$
\hat{y}_j =
\frac{e^{z_j}}
{\sum_{k=1}^{C} e^{z_k}}
$$

The loss function is cross-entropy:

$$
L(\hat{y}, y)
=
-\sum_{j=1}^{C} y_j \log(\hat{y}_j)
$$

The key gradient result is:

$$
dZ^{[L]} = A^{[L]} - Y
$$

The essence is:

$$
\boxed{
\text{Softmax converts logits into a probability distribution over multiple classes.}
}
$$

Softmax Regression is logistic regression generalized to multi-class classification, where the classes compete and the total probability is equal to $1$.

# Training a Softmax Classifier

## 1. Core Idea

A Softmax classifier is used for **multi-class single-label classification**, where each example belongs to exactly one class.

The model first produces raw scores called logits:

$$
Z^{[L]} = W^{[L]}A^{[L-1]} + b^{[L]}
$$

Then Softmax converts these logits into a probability distribution over $C$ classes:

$$
A^{[L]} = \text{softmax}(Z^{[L]})
$$

For class $j$:

$$
\hat{y}_j =
\frac{e^{z_j}}
{\sum_{k=1}^{C} e^{z_k}}
$$

The predicted class is usually:

$$
\hat{c} = \arg\max_j \hat{y}_j
$$

The key idea is:

$$
\boxed{
\text{Softmax turns logits into class probabilities.}
}
$$

## 2. Loss Function: Cross-Entropy

Softmax is usually trained with cross-entropy loss.

For one example:

$$
L(\hat{y}, y)
=
-\sum_{j=1}^{C} y_j \log(\hat{y}_j)
$$

If $y$ is one-hot and the correct class is $c$, then only the correct class term remains:

$$
L(\hat{y}, y) = -\log(\hat{y}_c)
$$

So if the model assigns high probability to the correct class, the loss is small. If it assigns low probability to the correct class, the loss is large.

For $m$ examples:

$$
J =
-\frac{1}{m}
\sum_{i=1}^{m}
\sum_{j=1}^{C}
y_j^{(i)}\log(\hat{y}_j^{(i)})
$$

## 3. Backward Propagation

The most important result when training a Softmax classifier is:

$$
dZ^{[L]} = A^{[L]} - Y
$$

This means:

$$
\text{gradient} = \text{predicted probabilities} - \text{true labels}
$$

For example, if:

$$
A^{[L]}
=
\begin{bmatrix}
0.05 \\
0.80 \\
0.15
\end{bmatrix}
$$

and:

$$
Y =
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

then:

$$
dZ^{[L]}
=
A^{[L]} - Y
=
\begin{bmatrix}
0.05 \\
-0.20 \\
0.15
\end{bmatrix}
$$

This gradient tells the model to reduce logits of incorrect classes and increase the logit of the correct class.

After computing $dZ^{[L]}$, the output layer parameters are updated through the usual backpropagation process:

$$
W^{[L]} := W^{[L]} - \alpha dW^{[L]}
$$

$$
b^{[L]} := b^{[L]} - \alpha db^{[L]}
$$

## 4. Training Procedure

Training a Softmax classifier follows the usual neural network training loop:

```text
1. Compute logits.
2. Apply Softmax to get class probabilities.
3. Compute cross-entropy loss.
4. Backpropagate gradients.
5. Update parameters.
6. Repeat for many iterations or epochs.
```

In compact form:

$$
Z^{[L]}
\rightarrow
A^{[L]} = \text{softmax}(Z^{[L]})
\rightarrow
J = \text{cross-entropy}(A^{[L]},Y)
\rightarrow
\text{backpropagation}
\rightarrow
\text{parameter update}
$$

## 5. Numerical Stability

When implementing Softmax manually, computing $e^{z_j}$ directly can cause overflow if logits are large.

A stable implementation subtracts the maximum logit first:

$$
\text{softmax}(z)_j =
\frac{e^{z_j-\max(z)}}
{\sum_k e^{z_k-\max(z)}}
$$

This does not change the Softmax output, but it makes computation safer.

Minimal NumPy implementation:

~~~python
import numpy as np

def softmax(Z):
    Z_shifted = Z - np.max(Z, axis=1, keepdims=True)
    exp_Z = np.exp(Z_shifted)
    return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)
~~~

## 6. PyTorch and TensorFlow Notes

In PyTorch, `nn.CrossEntropyLoss()` expects **raw logits**, not Softmax probabilities.

Correct usage:

~~~python
import torch.nn as nn

logits = model(X)
loss = nn.CrossEntropyLoss()(logits, y)
~~~

Do not apply Softmax before `CrossEntropyLoss`, because PyTorch already combines `LogSoftmax` and negative log-likelihood internally.

In TensorFlow/Keras, if the model outputs logits, use:

~~~python
import tensorflow as tf

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
~~~

If the final layer already uses Softmax, then use:

~~~python
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
~~~

The important rule is:

```text
Output logits → from_logits=True
Output probabilities → from_logits=False
```

## 7. Softmax vs Sigmoid

Softmax is used for **multi-class single-label classification**.

Example:

```text
An image is either cat, dog, or horse.
```

The classes compete with each other, and probabilities sum to $1$.

Sigmoid is used for **multi-label classification**.

Example:

```text
An image may contain person, car, and dog at the same time.
```

Each class is treated as an independent binary classification problem.

The rule is:

```text
Multi-class single-label → Softmax
Multi-label → Sigmoid per class
```

## 8. Important Mistakes

A common mistake is applying Softmax before `nn.CrossEntropyLoss()` in PyTorch. This is unnecessary and can reduce numerical stability.

Another mistake is using Softmax for multi-label classification. Softmax assumes that exactly one class is correct.

Another important mistake is using the wrong label format. In PyTorch, `CrossEntropyLoss` usually expects class indices, not one-hot vectors.

For example:

```text
logits shape: (m, C)
labels shape: (m,)
```

where each label is a class index.

## 9. Essential Conclusion

Training a Softmax classifier means combining:

$$
\text{logits} + \text{softmax} + \text{cross-entropy} + \text{backpropagation}
$$

Softmax converts logits into a probability distribution:

$$
\hat{y}_j =
\frac{e^{z_j}}
{\sum_{k=1}^{C} e^{z_k}}
$$

Cross-entropy measures how much probability the model assigns to the correct class:

$$
L(\hat{y}, y) = -\log(\hat{y}_c)
$$

The key gradient is:

$$
dZ^{[L]} = A^{[L]} - Y
$$

The essence is:

$$
\boxed{
\text{Training a Softmax classifier increases the probability of the correct class and decreases the probabilities of incorrect classes.}
}
$$

# Deep Learning Programming Frameworks

## 1. Core Idea

Deep learning frameworks such as **PyTorch** and **TensorFlow/Keras** help us build, train, and evaluate neural networks more efficiently.

When implementing neural networks from scratch with NumPy, we need to manually write:

```text
Forward propagation
Loss computation
Backward propagation
Gradient calculation
Parameter updates
Optimizers
Dropout
Batch Normalization
Softmax and Cross-Entropy
```

This is useful for understanding the theory, but it is error-prone and inefficient for real projects.

A framework automates many of these steps, especially:

$$
\text{gradient computation}
$$

and:

$$
\text{parameter updates}
$$

The key idea is:

$$
\boxed{
\text{NumPy helps us understand neural networks. Frameworks help us build them in practice.}
}
$$

## 2. What Frameworks Provide

A deep learning framework usually provides several core components.

First, it provides tensor operations. Tensors are similar to NumPy arrays, but they can run efficiently on GPU or TPU.

Second, it provides ready-to-use layers such as:

```text
Linear / Dense
Convolution
BatchNorm
Dropout
Activation functions
RNN / LSTM / Transformer layers
```

Third, it provides common loss functions such as:

```text
MSELoss
Binary Cross-Entropy
Cross-Entropy Loss
Sparse Categorical Cross-Entropy
```

Fourth, it provides optimizers such as:

```text
SGD
Momentum
RMSprop
Adam
AdamW
```

Most importantly, frameworks provide **automatic differentiation**. This means we write the forward computation and the loss, then the framework automatically computes gradients through backpropagation.

In PyTorch, this is done by:

~~~python
loss.backward()
~~~

Then the optimizer updates the parameters:

~~~python
optimizer.step()
~~~

## 3. Computation Graph and Autograd

A framework represents forward propagation as a computation graph.

For example:

$$
Z = XW + b
$$

$$
A = g(Z)
$$

$$
J = L(A,Y)
$$

The graph is:

```text
X, W, b → Z → A → J
```

During backward propagation, the framework traverses this graph in reverse:

```text
J → A → Z → W, b
```

This is how the framework automatically computes gradients such as:

$$
dW,\quad db
$$

In NumPy, we have to derive and implement these gradients manually. In PyTorch or TensorFlow, the framework handles this through autograd.

However, autograd does not remove the need to understand the math. We still need to know whether the model output, loss function, labels, and tensor shapes are correct.

## 4. Standard Training Workflow

A standard deep learning workflow with a framework is:

```text
1. Define the model.
2. Choose the loss function.
3. Choose the optimizer.
4. Run forward propagation.
5. Compute the loss.
6. Run backward propagation.
7. Update parameters.
8. Evaluate the model correctly.
```

In PyTorch, a minimal training loop looks like:

~~~python
for X_batch, y_batch in train_loader:
    logits = model(X_batch)
    loss = loss_fn(logits, y_batch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
~~~

The meaning of each step is:

```text
logits = model(X_batch)
```

runs forward propagation.

```text
loss = loss_fn(logits, y_batch)
```

computes the training objective.

```text
optimizer.zero_grad()
```

clears old gradients.

```text
loss.backward()
```

computes gradients by backpropagation.

```text
optimizer.step()
```

updates model parameters.

The line `optimizer.zero_grad()` is important because PyTorch accumulates gradients by default.

## 5. PyTorch Example

A simple multi-class classifier in PyTorch can be written as:

~~~python
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)
~~~

The final layer returns logits, not Softmax probabilities.

For multi-class classification, we use:

~~~python
loss_fn = nn.CrossEntropyLoss()
~~~

Important point:

> `nn.CrossEntropyLoss()` expects raw logits, not probabilities after Softmax.

Correct:

~~~python
logits = model(X)
loss = nn.CrossEntropyLoss()(logits, y)
~~~

Incorrect:

~~~python
probs = torch.softmax(model(X), dim=1)
loss = nn.CrossEntropyLoss()(probs, y)
~~~

PyTorch already combines `LogSoftmax` and negative log-likelihood internally in a numerically stable way.

## 6. TensorFlow/Keras Example

In TensorFlow/Keras, the workflow can be more compact.

A simple classifier can be written as:

~~~python
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(num_classes)
])
~~~

If the final layer returns logits, the loss should use:

~~~python
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
~~~

Then:

~~~python
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=loss_fn,
    metrics=["accuracy"]
)

model.fit(X_train, y_train, epochs=10, batch_size=64)
~~~

If the final layer already uses Softmax:

~~~python
tf.keras.layers.Dense(num_classes, activation="softmax")
~~~

then the loss should use:

~~~python
from_logits=False
~~~

The rule is:

```text
Output logits → from_logits=True
Output probabilities → from_logits=False
```

## 7. Training Mode and Evaluation Mode

Frameworks also manage layers that behave differently during training and testing.

Two important examples are:

```text
Dropout
BatchNorm
```

During training:

```text
Dropout randomly drops activations.
BatchNorm uses mini-batch statistics.
```

During evaluation:

```text
Dropout is turned off.
BatchNorm uses running statistics.
```

In PyTorch, we must explicitly switch modes:

~~~python
model.train()
~~~

for training, and:

~~~python
model.eval()
~~~

for validation or testing.

During evaluation, we should also use:

~~~python
with torch.no_grad():
    logits = model(X_test)
~~~

This avoids building a computation graph, saving memory and computation.

A common mistake is forgetting `model.eval()` during validation or testing, which can make results unstable when the model contains Dropout or BatchNorm.

## 8. Relationship to Week 3 Topics

Frameworks directly support the main techniques from Course 2 Week 3.

For hyperparameter tuning, frameworks make it easy to change:

```text
learning rate
batch size
number of layers
hidden units
dropout rate
weight decay
BatchNorm on/off
optimizer type
```

For BatchNorm, instead of manually implementing:

$$
Z_{\text{norm}}
=
\frac{Z-\mu}{\sqrt{\sigma^2+\epsilon}}
$$

we can use:

~~~python
nn.BatchNorm1d(hidden_dim)
~~~

For Softmax classification, instead of manually combining Softmax and Cross-Entropy, we can use:

~~~python
nn.CrossEntropyLoss()
~~~

For optimization, instead of implementing Adam from scratch, we can use:

~~~python
torch.optim.Adam(model.parameters(), lr=0.001)
~~~

Frameworks allow us to focus more on architecture, debugging, experiments, and analysis.

## 9. Common Mistakes

A common mistake is relying on the framework without understanding the math.

For example, using Softmax before `CrossEntropyLoss` in PyTorch may run without an error, but it is conceptually wrong.

Another common mistake is forgetting to clear gradients:

~~~python
optimizer.zero_grad()
~~~

If this is skipped, gradients accumulate across batches.

Another mistake is using the wrong label format. In PyTorch, for `CrossEntropyLoss`, the usual format is:

```text
logits shape: (m, C)
labels shape: (m,)
```

where labels are class indices, not one-hot vectors.

Another mistake is forgetting:

~~~python
model.eval()
~~~

during validation or testing.

A final common mistake is confusing logits and probabilities. Some loss functions expect logits, while others expect probabilities. This must be checked carefully.

## 10. Essential Conclusion

Deep learning frameworks automate the repetitive and error-prone parts of neural network training.

They provide:

```text
tensor operations
layers
loss functions
optimizers
automatic differentiation
GPU/TPU acceleration
training and evaluation utilities
```

But frameworks do not replace conceptual understanding.

A strong deep learning engineer still needs to understand:

```text
forward propagation
loss functions
backpropagation
optimizer behavior
tensor shapes
training vs evaluation mode
overfitting and underfitting
```

The essence is:

$$
\boxed{
\text{Frameworks automate computation, but they do not replace understanding.}
}
$$

NumPy is useful for learning how neural networks work internally. Frameworks are essential for building real deep learning systems efficiently, reliably, and at scale.